[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/martinloretzzz/vector-index-embedding/blob/main/quickstart.ipynb)

# Accelerating LLM Inference via Vector Index Based Output Embeddings

This notebook demonstrates how to accelerate LLM inference for compact models using vector index-based output embeddings, as described in our paper.

[Arxiv Link (online soon)]()

## Abstract

Large output embedding matrices create a significant memory bandwidth bottleneck during autoregressive decoding, especially for compact LLMs
with large multilingual vocabularies. We reformulate the output projection followed by top-k token
selection as a maximum inner product search over
token embeddings and replace the dense vocabulary projection with an HNSW-based vector index.
The resulting output head retrieves only a small
candidate set of high-scoring tokens and can be
integrated into existing decoding pipelines by scattering retrieved logits into a sparse full-vocabulary
tensor. On CPU inference with Gemma 3, Llama
3.2, and Qwen 3 models, our method substantially
accelerates the output projection and improves
end-to-end batch-size-one decoding throughput
by up to 82% for Gemma 3 270M, while preserving generation quality under AlpacaEval evaluation. These results suggest approximate retrieval
is a practical alternative to dense output projections in latency-sensitive small-batch decoding.

In [1]:
!pip install -q transformers vector-index-embedding

## Usage

Using the vector index embedding is straightforward and integrates seamlessly with Hugging Face transformers by simply replacing the model's lm_head. To try it out immediately, we provide prebuilt vector indices for several popular models on [Hugging Face](https://huggingface.co/martinloretzzz/vector-index-embedding).

In [2]:
from transformers import pipeline
from vectorindex import VectorIndexEmbedding

model_id = "Qwen/Qwen3-0.6B" # "google/gemma-3-270m-it"
pipe = pipeline("text-generation", model=model_id, device="cpu")
pipe.model.lm_head = VectorIndexEmbedding.from_pretrained(model_id, ef=200)

prompt = [{"role": "user", "content": "Who was Alan Turing?"}]
out = pipe(prompt)
print(out[0]["generated_text"][-1]["content"])

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

qwen-qwen3-0-6b.index:   0%|          | 0.00/664M [00:00<?, ?B/s]

qwen-qwen3-0-6b.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<think>
Okay, the user asked "Who was Alan Turing?" I need to provide a clear and accurate answer. Let me start by recalling the basics.

Alan Turing was a British computer scientist and mathematician. He made significant contributions to the field of computer science. I should mention his key works and inventions, like the ENIAC and the concept of Turing machines. Also, his role in breaking the Enigma code is important. I need to highlight his impact on modern computing and the field of artificial intelligence.

Wait, did he work on the Turing Test? Yes, that's a key part. Also, his death in 1950 is important. I should mention his legacy and how his work influenced today's technology. Make sure the answer is concise but covers all important points. Check for any possible errors, like dates or inventions. No, I think that's correct. Alright, time to put it all together in a clear structure.
</think>

Alan Turing was a British computer scientist and mathematician who made groundbreaking

## Performance & Accuracy

We compare the performance and generated text of our vector index model to the unmodified baseline. You can increase the ef parameter for better recall, but higher search times. To get even better performance, you can use our modified version of hnswlib [martinloretzzz/hnswlib](https://github.com/martinloretzzz/hnswlib), but that might not work on all systems, as that code was only built and tested on our benchmarking server.

In [3]:
from transformers import pipeline
from vectorindex import VectorIndexEmbedding
import time
import torch

model_id = "google/gemma-3-270m-it" # "Qwen/Qwen3-0.6B"
new_token_count = 256
prompt = [{"role": "user", "content": "How does a computer work?"}]

pipe_ref = pipeline("text-generation", model=model_id, device="cpu", dtype=torch.float32)
pipe_ref.model.generation_config.cache_implementation = "static"
pipe_ref.model.forward = torch.compile(pipe_ref.model.forward, mode="reduce-overhead", backend="inductor")

pipe_vec = pipeline("text-generation", model=model_id, device="cpu", dtype=torch.float32)
pipe_vec.model.lm_head = VectorIndexEmbedding.from_pretrained(model_id, ef=200)
pipe_vec.model.generation_config.cache_implementation = "static"
pipe_vec.model.forward = torch.compile(pipe_vec.model.forward, mode="reduce-overhead", backend="inductor")


def timeit_wrapper(func, num_warmup=1, num_run=2):
    out = None
    for i in range(num_warmup):
        out_warmup = func()
    start = time.perf_counter()
    for i in range(num_run):
        out = func()
    end = time.perf_counter()
    t = (end - start) / num_run
    return out, t

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

google-gemma-3-270m-it.index:   0%|          | 0.00/743M [00:00<?, ?B/s]

google-gemma-3-270m-it.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

In [4]:
# Baseline model
out, time_ref = timeit_wrapper(lambda: pipe_ref(prompt, max_new_tokens=new_token_count, min_new_tokens=new_token_count, do_sample=False, eos_token_id=None, pad_token_id=None))
text_ref = out[0]["generated_text"][1]["content"]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'eos_token_id', 'min_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=256) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is desi

In [5]:
# Vector Index Embedding Model
out, time_vec = timeit_wrapper(lambda: pipe_vec(prompt, max_new_tokens=new_token_count, min_new_tokens=new_token_count, do_sample=False, eos_token_id=None, pad_token_id=None))
text_vec = out[0]["generated_text"][1]["content"]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=256) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=256) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/do

In [6]:
print(f"Baseline Model:               time = {time_ref:.2f}s throughput = {new_token_count / time_ref:.2f} tok/s")
print(f"Vector Index Embedding Model: time = {time_vec:.2f}s throughput = {new_token_count / time_vec:.2f} tok/s")
print(f"Speedup: {time_ref / time_vec:.4f}x")

Baseline Model:               time = 73.51s throughput = 3.48 tok/s
Vector Index Embedding Model: time = 46.38s throughput = 5.52 tok/s
Speedup: 1.5850x


In [7]:
print("Text Baseline:")
print(text_ref)

print("\n\n")

print("Text Vector Index Embedding Model:")
print(text_vec)

Text Baseline:
The operation of a computer is a complex process involving a network of interconnected components, all working together to perform specific tasks. Here's a breakdown of the key components and how they work:

**1. Central Processing Unit (CPU):**

*   **Function:** The CPU is the "brain" of the computer. It executes instructions, which are the instructions that tell the computer what to do.
*   **How it works:** The CPU fetches instructions from memory, decodes them, and then executes them. It also manages the overall flow of data and logic.
*   **Key Components:**
    *   **Arithmetic Logic Unit (ALU):** Performs arithmetic and logical operations.
    *   **Control Unit:** Manages the flow of data and logic, making decisions.
    *   **Registers:** Small, fast storage locations within the CPU that hold the data and instructions that are currently being processed.

**2. Memory (RAM):**

*   **Function:** RAM is the computer's main storage area. It holds the data and instr

## Build a vector index embedding for a model

If we do not provide a prebuilt index for your desired model, you can easily generate one yourself. While the construction process takes some time, it only needs to be performed once, and the resulting index can be distributed alongside the model.
Below is our default index configuration when used with ef_construction = 5000. You can experiment with these parameters: higher values generally result in higher-quality indices but increase search latency.
Because approximate search might occasionally miss critical special tokens—such as the EOF token that terminates generation, it is recommended to add these tokens explicitly to the special_tokens list to have them always evaluated. The generated indices are saved to the data subfolder and can be loaded via: `VectorIndexEmbedding.from_file(f"data/<index_id>.index", ef=200)

In [8]:
model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"

# The amount of edges in the HNSW graph.
# 32 is a good tradeoff, higher leads to higher quality index, but slower search time.
M = 32

# Size of the candidate list during index construction.
# For production indices we use 5000 by default, which is very slow, but we only construct them once.
# For testing here, we just use 200 to get quick results.
ef_construction = 200

# Size of the candidate list evaluated during search.
ef = 200

# The number of nearest tokens used for top-k sampling.
k = 50

In [9]:
from vectorindex import VectorIndexEmbedding
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from vectorindex.vector_index import VectorIndexEmbeddingConfig

def get_special_tokens(model_id, tokenizer):
    add_tokenizer_special_tokens = True
    extra_tokens = []

    if "llama" in model_id:
        extra_tokens = ["'s", 'Ġand', 'Ġ**', ',', 'Ġto', ':**']
    if "Qwen3" in model_id:
        extra_tokens = ["\\n", "Ċ", "ĊĊ", ",", ".", "Ġ", "Ġor", "Ġto", "Ġand", "Ġa", "Ġof", "Ġ**"]
    if "google" in model_id:
        add_tokenizer_special_tokens = False
        extra_tokens = ["-", "▁of", "▁and", "▁the", "▁to", "▁a", ".", "▁▁", ",", "'", '"', ":**", "▁**", "\n", '▁"', "<pad>", "<eos>", "<bos>", "<unk>", "<mask>", "[multimodal]", "<start_of_turn>", "<end_of_turn>"]

    special_tokens = []
    if add_tokenizer_special_tokens:
        special_tokens.extend(list([k for k, v in tokenizer.added_tokens_decoder.items() if not "unused" in v.content and not "img_row" in v.content and not "reserved" in v.content and k < vocab_size]))
    special_tokens.extend([tokenizer.vocab[t] for t in extra_tokens])

    print(special_tokens)
    return special_tokens

set_seed(42)

model = AutoModelForCausalLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

weight = model.lm_head.weight.detach().clone().float()
vocab_size = weight.shape[0]

special_tokens = get_special_tokens(model_id, tokenizer)

index_id = VectorIndexEmbedding.get_index_name(model_id)
config = VectorIndexEmbeddingConfig(model_id=model_id, model_name=index_id, k=k, M=M, ef=ef, ef_construction=ef_construction, special_tokens=special_tokens)
print(f"Build index for config: {config}")
VectorIndexEmbedding.build_index(weight, config, save_path="data")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
Build index for config: VectorIndexEmbeddingConfig(model_name='huggingfacetb-smollm2-360m-instruct', k=50, ef=200, M=32, ef_construction=200, special_tokens=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], dim=-1, vocab_size=-1, model_id='HuggingFaceTB/SmolLM2-360M-Instruct')
Index saved to data/huggingfacetb-smollm2-360m-instruct.index


'data/huggingfacetb-smollm2-360m-instruct.index'

In [11]:
from transformers import pipeline
from vectorindex import VectorIndexEmbedding

pipe = pipeline("text-generation", model=model_id, device="cpu")
pipe.model.lm_head = VectorIndexEmbedding.from_file(f"data/{VectorIndexEmbedding.get_index_name(model_id)}.index", ef=200)

prompt = [{"role": "user", "content": "What was at the beginning of time?"}]
out = pipe(prompt)
print(out[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I'm a chat AI, I don't have personal experiences or emotions like humans do. However, I can provide a general answer based on current scientific theories and literature.

The universe began with a Big-Bang event approximately 13.8 billion years back in the universe's early stages. This event is thought to have taken place around 13.8 billion years before the current date. After the Big-Bang, the universe expanded rapidly, and the initial conditions for the formation of structures such as black-hits were set. 

The universe then began to cool and expand, eventually forming the Big-Bang Singularity, which was a singularity or a point of infinite mass and energy, that expanded into a multiverse, creating the four known fundamental forces of nature: the strong, weak, and the electric forces.

The universe then evolved through different stages such as the first few seconds of the Big-Bang, the formation of the first sub-stance, and the evolution of life.
